# Module 01 Assignment: Python for Data Science

**6 tasks | Auto-graded with assert tests**

Complete each task by replacing `# YOUR CODE HERE` with working code.
Run the assert cell immediately below your solution to check your work.
All tasks use the same synthetic NYC 311 dataset generated in Task 0.

> **Do not modify the assert cells.**

In [ ]:
# ── Setup (run this first) ──────────────────────────────────────────────────
import numpy as np
import pandas as pd

np.random.seed(0)

N = 5_000
boroughs = ['BROOKLYN', 'QUEENS', 'MANHATTAN', 'BRONX', 'STATEN ISLAND']
b_weights = [0.28, 0.24, 0.23, 0.19, 0.06]

complaint_types = [
    'Noise - Residential', 'HEAT/HOT WATER', 'Illegal Parking',
    'Blocked Driveway', 'Street Light Condition', 'Noise - Street/Sidewalk',
    'UNSANITARY CONDITION', 'Rodent', 'Graffiti', 'Water System'
]
c_weights = [0.18, 0.14, 0.12, 0.10, 0.09, 0.09, 0.08, 0.07, 0.07, 0.06]

created = pd.date_range('2023-01-01', periods=N, freq='2h')
res_hrs = np.abs(np.random.normal(60, 40, N)).clip(1, 500)
closed = created + pd.to_timedelta(res_hrs, unit='h')
has_close = np.random.choice([True, False], N, p=[0.82, 0.18])

df = pd.DataFrame({
    'Unique Key': range(1, N + 1),
    'Created Date': created,
    'Closed Date': np.where(has_close, closed, pd.NaT),
    'Complaint Type': np.random.choice(complaint_types, N, p=c_weights),
    'Borough': np.random.choice(boroughs, N, p=b_weights),
})

# Introduce messy borough values
mess_idx = np.random.choice(df.index, 200, replace=False)
df.loc[mess_idx[:100], 'Borough'] = 'Unspecified'
df.loc[mess_idx[100:], 'Borough'] = np.nan

print(f'Dataset ready: {df.shape}')
df.head(3)

---
## Task 1 — Count Missing Values (Recall)

Create a Series called `missing_counts` that contains the number of missing
values in *each column* of `df`, **including** columns with zero missing values.

*Hint: combine `.isnull()` and `.sum()`.*

In [ ]:
# YOUR CODE HERE
missing_counts = ...

In [ ]:
# ── Assertion checks ───────────────────────────────────────────────────────
assert isinstance(missing_counts, pd.Series), \
    f"missing_counts should be a pd.Series, got {type(missing_counts)}"
assert set(missing_counts.index) == set(df.columns), \
    "missing_counts must have an entry for every column in df"
assert missing_counts['Unique Key'] == 0, \
    "'Unique Key' has no missing values — expected 0"
assert missing_counts['Borough'] == 100, \
    f"'Borough' should have 100 NaN rows, got {missing_counts['Borough']}"
assert missing_counts['Closed Date'] > 0, \
    "'Closed Date' should have some NaT values"
print('Task 1 passed!')

---
## Task 2 — Clean Borough Names (Recall)

Create a new column `'Borough_clean'` in `df` where:
- NaN values become the string `'Unknown'`
- `'Unspecified'` (any case) becomes `'Unknown'`
- All other values are converted to **Title Case** (e.g. `'BROOKLYN'` → `'Brooklyn'`)

Do **not** modify the original `'Borough'` column.

In [ ]:
# YOUR CODE HERE
def clean_borough(val):
    ...

df['Borough_clean'] = ...

In [ ]:
# ── Assertion checks ───────────────────────────────────────────────────────
assert 'Borough_clean' in df.columns, \
    "Column 'Borough_clean' not found in df"
assert df['Borough_clean'].isnull().sum() == 0, \
    "'Borough_clean' should have no missing values"
assert 'Unknown' in df['Borough_clean'].values, \
    "Expected 'Unknown' in Borough_clean for NaN/Unspecified rows"
assert df['Borough_clean'].str.contains('BROOKLYN').sum() == 0, \
    "Borough names should be Title Case, not ALL CAPS"
assert 'Brooklyn' in df['Borough_clean'].values, \
    "Expected 'Brooklyn' (Title Case) in Borough_clean"
unknown_count = (df['Borough_clean'] == 'Unknown').sum()
assert unknown_count == 200, \
    f"Expected 200 'Unknown' rows (100 NaN + 100 Unspecified), got {unknown_count}"
print('Task 2 passed!')

---
## Task 3 — GroupBy Aggregation (Application)

Using the cleaned `'Borough_clean'` column, create a DataFrame called
`borough_summary` that, **excluding 'Unknown' boroughs**, contains:

| Column | Description |
|--------|-------------|
| `complaint_count` | Total number of complaints per borough |
| `pct_of_total` | Each borough's share of all non-Unknown complaints (0–100, rounded to 1 dp) |

Sort by `complaint_count` descending. Index should be borough names.

In [ ]:
# YOUR CODE HERE
borough_summary = ...

In [ ]:
# ── Assertion checks ───────────────────────────────────────────────────────
assert isinstance(borough_summary, pd.DataFrame), \
    f"borough_summary should be a DataFrame, got {type(borough_summary)}"
assert 'complaint_count' in borough_summary.columns, \
    "Missing column 'complaint_count'"
assert 'pct_of_total' in borough_summary.columns, \
    "Missing column 'pct_of_total'"
assert 'Unknown' not in borough_summary.index, \
    "'Unknown' borough should be excluded from borough_summary"
assert len(borough_summary) == 5, \
    f"Expected 5 boroughs, got {len(borough_summary)}"
assert borough_summary.index[0] == 'Brooklyn', \
    f"Expected 'Brooklyn' as top borough, got {borough_summary.index[0]}"
total_pct = borough_summary['pct_of_total'].sum()
assert abs(total_pct - 100.0) < 0.5, \
    f"pct_of_total should sum to ~100, got {total_pct:.1f}"
print('Task 3 passed!')

---
## Task 4 — Date Feature Engineering (Application)

Add two new columns to `df`:

1. `'day_of_week'` — the integer day-of-week where **Monday = 0, Sunday = 6**,
   derived from `'Created Date'`.
2. `'resolution_hours'` — the time between `'Created Date'` and `'Closed Date'`
   expressed in **hours** (float). Rows without a `'Closed Date'` should have
   `NaN` for `resolution_hours`.

*Hint: use `.dt` accessors and `timedelta` arithmetic.*

In [ ]:
# YOUR CODE HERE
df['day_of_week'] = ...
df['resolution_hours'] = ...

In [ ]:
# ── Assertion checks ───────────────────────────────────────────────────────
assert 'day_of_week' in df.columns, "Column 'day_of_week' not found"
assert 'resolution_hours' in df.columns, "Column 'resolution_hours' not found"
assert df['day_of_week'].min() == 0, \
    f"Min day_of_week should be 0 (Monday), got {df['day_of_week'].min()}"
assert df['day_of_week'].max() == 6, \
    f"Max day_of_week should be 6 (Sunday), got {df['day_of_week'].max()}"
assert df['resolution_hours'].isnull().sum() > 0, \
    "Expected NaN in resolution_hours for rows without Closed Date"
assert (df['resolution_hours'].dropna() > 0).all(), \
    "All resolution times should be positive"
# Spot-check: row 0 should have a closed date, so resolution_hours must be set
first_closed = df.loc[df['Closed Date'].notna()].iloc[0]
expected_hrs = (first_closed['Closed Date'] - first_closed['Created Date']).total_seconds() / 3600
actual_hrs = first_closed['resolution_hours']
assert abs(actual_hrs - expected_hrs) < 0.01, \
    f"resolution_hours calculation off: expected {expected_hrs:.2f}, got {actual_hrs:.2f}"
print('Task 4 passed!')

---
## Task 5 — Top Complaint by Borough (Synthesis)

Create a Series called `top_complaint_by_borough` where:
- The **index** is the five valid borough names (excluding 'Unknown').
- The **values** are the most common `'Complaint Type'` in that borough.

If two complaint types are tied, return either one (consistent with
`value_counts().idxmax()` behaviour).

*Hint: consider using `groupby` + `apply` with a lambda, or `groupby` +
`value_counts` with `groupby(...).transform`.*

In [ ]:
# YOUR CODE HERE
top_complaint_by_borough = ...

In [ ]:
# ── Assertion checks ───────────────────────────────────────────────────────
assert isinstance(top_complaint_by_borough, pd.Series), \
    f"Expected pd.Series, got {type(top_complaint_by_borough)}"
assert len(top_complaint_by_borough) == 5, \
    f"Expected 5 boroughs, got {len(top_complaint_by_borough)}"
assert 'Unknown' not in top_complaint_by_borough.index, \
    "'Unknown' should not be in the result"
expected_boroughs = {'Brooklyn', 'Queens', 'Manhattan', 'Bronx', 'Staten Island'}
assert set(top_complaint_by_borough.index) == expected_boroughs, \
    f"Index boroughs don't match: {set(top_complaint_by_borough.index)}"
# Validate one borough manually
bk_top = (
    df.loc[df['Borough_clean'] == 'Brooklyn', 'Complaint Type']
    .value_counts().idxmax()
)
assert top_complaint_by_borough['Brooklyn'] == bk_top, \
    f"Brooklyn top complaint: expected '{bk_top}', got '{top_complaint_by_borough['Brooklyn']}'"
print('Task 5 passed!')

---
## Task 6 — Vectorised NumPy Analysis (Synthesis/Analysis)

Using **NumPy only** (no Pandas methods), compute the following from the
`resolution_hours` column values (drop NaN first):

- `mean_resolution` — arithmetic mean (float, rounded to 2 dp)
- `pct_under_24h` — percentage of complaints resolved in under 24 hours
  (float, rounded to 1 dp, range 0–100)
- `resolution_std` — standard deviation (float, rounded to 2 dp)

*Hint: extract the array with `.dropna().values`, then use `np.mean`,
`np.std`, and boolean masking.*

In [ ]:
# YOUR CODE HERE
hours = df['resolution_hours'].dropna().values  # NumPy array

mean_resolution = ...
pct_under_24h = ...
resolution_std = ...

print(f'Mean resolution: {mean_resolution} hrs')
print(f'Resolved < 24 h: {pct_under_24h}%')
print(f'Std deviation:   {resolution_std} hrs')

In [ ]:
# ── Assertion checks ───────────────────────────────────────────────────────
hours_check = df['resolution_hours'].dropna().values

assert isinstance(mean_resolution, float), \
    f"mean_resolution should be float, got {type(mean_resolution)}"
assert abs(mean_resolution - round(float(np.mean(hours_check)), 2)) < 0.01, \
    f"mean_resolution incorrect: expected {round(float(np.mean(hours_check)), 2)}, got {mean_resolution}"

expected_pct = round(float((hours_check < 24).mean() * 100), 1)
assert abs(pct_under_24h - expected_pct) < 0.2, \
    f"pct_under_24h incorrect: expected {expected_pct}, got {pct_under_24h}"
assert 0 <= pct_under_24h <= 100, \
    f"pct_under_24h should be between 0 and 100, got {pct_under_24h}"

expected_std = round(float(np.std(hours_check)), 2)
assert abs(resolution_std - expected_std) < 0.1, \
    f"resolution_std incorrect: expected {expected_std}, got {resolution_std}"

print('Task 6 passed!')
print('\n🎉 All 6 tasks complete! Assignment done.')